用jieba
max∑logP(w1,w2,...,wn)
动态规划划分
创建DAG，从后向前遍历



In [2]:
import jieba

text = "我来到北京清华大学"
seg_list = jieba.lcut(text, cut_all=False)
print(seg_list)

Building prefix dict from the default dictionary ...
Dumping model to file cache C:\Users\zhuxu\AppData\Local\Temp\jieba.cache
Loading model cost 0.833 seconds.
Prefix dict has been built successfully.


['我', '来到', '北京', '清华大学']


In [1]:
import jieba
text = "九头虫让奔波儿灞把唐僧师徒除掉"
seg_list = jieba.lcut(text, cut_all=False)
print(seg_list)

seg_list = jieba.lcut(text, HMM=True)
print(seg_list)

#jieba.load_userdict("./user_dict.txt")
seg_list = jieba.lcut(text, cut_all=False)
print(seg_list)

d:\deeplearning\.venv\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\zhuxu\AppData\Local\Temp\jieba.cache
Loading model cost 0.722 seconds.
Prefix dict has been built successfully.


['九头', '虫', '让', '奔波', '儿', '灞', '把', '唐僧', '师徒', '除掉']
['九头', '虫', '让', '奔波', '儿', '灞', '把', '唐僧', '师徒', '除掉']
['九头', '虫', '让', '奔波', '儿', '灞', '把', '唐僧', '师徒', '除掉']


隐马尔科夫模型（HMM）
状态：B（词首），M（词中），E（词尾），S（单字成词）
观测：字
jieba中加上HMM时
用缓存库存单词，结束或遇见多词时，遍历缓存库，尝试找出未识别词


In [5]:
import jieba
text = "我在boss直聘看到了一个不错的职位"
seg_list = jieba.lcut(text, HMM=True)
print(seg_list)

seg_list = jieba.lcut(text, HMM=False)
print(seg_list)

['我', '在', 'boss', '直聘', '看到', '了', '一个', '不错', '的', '职位']
['我', '在', 'boss', '直', '聘', '看到', '了', '一个', '不错', '的', '职位']


词性标注
用jieba.posseg模块

In [2]:
import jieba.posseg as pseg
text = "九头虫让奔波儿灞把唐僧师徒除掉"
words = pseg.lcut(text, HMM=True)
print(words)

[pair('九头', 'm'), pair('虫', 'n'), pair('让', 'v'), pair('奔波', 'v'), pair('儿', 'n'), pair('灞', 'x'), pair('把', 'p'), pair('唐僧', 'nr'), pair('师徒', 'n'), pair('除掉', 'v')]


# 主题模型
<p> 一篇文档由多个"主题"按一定比例混合而成，而一个主题又由多个"词语"按一定概率组成。 </p>
<p> 一个词的向量，就可以用它与各个主题的关联强度来表示。 </p>
<p><strong> Xm×n≈Wtopic,m×k  ×  Htopic,k×n</strong></p>
<p> W是词到k个主题的分布，而H是文档在K个主题上的分布。 </p>

# Word2Vec
<p> 根据上下文预测中心词 </p>
<p> 用稀疏向量乘稠密矩阵，相当于抽出该词，而矩阵就是词向量查询表 </p>

$$ \begin{bmatrix} 0 & 0 & 0 & \color{#42b983}{1} & 0 & 0 \end{bmatrix} \times \begin{bmatrix} 2 & 8 & 5 & 3 & 1 \\ 9 & 4 & 7 & 2 & 6 \\ 3 & 1 & 8 & 5 & 0 \\ 5 & 6 & 2 & 9 & 4 \\ 8 & 0 & 3 & 7 & 2 \\ 4 & 2 & 9 & 6 & 1 \end{bmatrix} $$

## CBOW模型
<p> 根据上下文词预测中心词 </p>

$$ \begin{aligned} \text{minimize } J &= -\log P(w_c | w_{c-m}, \ldots, w_{c-1}, w_{c+1}, \ldots, w_{c+m}) \ &= -\log P(u_c | h) \ &= -\log \frac{\exp(u_c^T h)}{\sum_{j=1}^{|V|} \exp(u_j^T h)} \ &= -u_c^T h + \log \sum_{j=1}^{|V|} \exp(u_j^T h) \end{aligned} $$

## Skip-Gram模型
<p> 根据中心词预测上下文词 </p>

$$ \begin{aligned} \text{minimize } J &= -\log P(w_{c-m}, \ldots, w_{c-1}, w_{c+1}, \ldots, w_{c+m} | w_c) \ &= -\log \prod_{j=0, j \neq m}^{2m} P(w_{c-m+j} | w_c) \ &= -\log \prod_{j=0, j \neq m}^{2m} P(u_{c-m+j} | v_c) \ &= -\log \prod_{j=0, j \neq m}^{2m} \frac{\exp(u_{c-m+j}^T v_c)}{\sum_{k=1}^{|V|} \exp(u_k^T v_c)} \ &= -\sum_{j=0, j \neq m}^{2m} u_{c-m+j}^T v_c + 2m \log \sum_{k=1}^{|V|} \exp(u_k^T v_c) \end{aligned} $$

## gensim工作流
<p> 1. 准备语料库（分词），并整理成Gensim要求的格式——一个由列表构成的列表 list[list[str]]，其中每个子列表代表一篇独立的文档。 </p>
<p> 2. 将分词映射到词ID </p>
<p> 3. 计算词袋 </p>


In [5]:
import jieba
from gensim import corpora

# 准备语料库
raw_headlines = ["央行降息，刺激股市反弹","球队赢得总决赛冠军，球员表现出色"]
tokenized_headlines = [jieba.lcut(doc) for doc in raw_headlines]
print (tokenized_headlines)

# 创建词典
dictionary = corpora.Dictionary(tokenized_headlines)
print (dictionary.token2id)

# 计算词袋
corpus = [dictionary.doc2bow(text) for text in tokenized_headlines]
print(corpus)

[['央行', '降息', '，', '刺激', '股市', '反弹'], ['球队', '赢得', '总决赛', '冠军', '，', '球员', '表现出色']]
{'刺激': 0, '反弹': 1, '央行': 2, '股市': 3, '降息': 4, '，': 5, '冠军': 6, '总决赛': 7, '球员': 8, '球队': 9, '表现出色': 10, '赢得': 11}
[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1)], [(5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1)]]


# TF-IDF

In [6]:
import jieba
from gensim import corpora,models

headlines = [
    "央行降息，刺激股市反弹",
    "球队赢得总决赛冠军，球员表现出色",
    "国家队公布最新一期足球集训名单",
    "A股市场持续震荡，投资者需谨慎",
    "篮球巨星刷新历史得分记录",
    "理财产品收益率创下新高"
]
tokenized_headlines = [jieba.lcut(doc) for doc in headlines]

dictionary = corpora.Dictionary(tokenized_headlines)
corpus_bow = [dictionary.doc2bow(doc) for doc in tokenized_headlines]

tfidf_model = models.TfidfModel(corpus_bow)#计算IDF值
corpus_tfidf = tfidf_model[corpus_bow]#计算TF-IDF值，返回的是（词ID，TF-IDF值）元组列表

def tfidf_with_word(tfidf_vec,id2word):
    pairs = [(id2word[id], value) for id, value in tfidf_vec]
    return sorted(pairs, key=lambda x: x[1], reverse=True)

first_tfidf = list(corpus_tfidf)[0]
print(first_tfidf)
print(tfidf_with_word(first_tfidf, dictionary))

new_headline = "新兴市场股市表现强劲"
new_tokenized = jieba.lcut(new_headline)
new_headline_bow = dictionary.doc2bow(new_tokenized)
new_headline_tfidf = tfidf_model[new_headline_bow]
print(new_headline_tfidf)
print(tfidf_with_word(new_headline_tfidf, dictionary))


[(0, np.float64(0.44066740566370055)), (1, np.float64(0.44066740566370055)), (2, np.float64(0.44066740566370055)), (3, np.float64(0.44066740566370055)), (4, np.float64(0.44066740566370055)), (5, np.float64(0.1704734229377651))]
[('刺激', np.float64(0.44066740566370055)), ('反弹', np.float64(0.44066740566370055)), ('央行', np.float64(0.44066740566370055)), ('股市', np.float64(0.44066740566370055)), ('降息', np.float64(0.44066740566370055)), ('，', np.float64(0.1704734229377651))]
[(3, np.float64(0.7071067811865476)), (20, np.float64(0.7071067811865476))]
[('股市', np.float64(0.7071067811865476)), ('市场', np.float64(0.7071067811865476))]


# LDA与主题挖掘

In [11]:
import jieba
from gensim import models,corpora
headlines = [
    "央行降息，刺激股市反弹",
    "球队赢得总决赛冠军，球员表现出色",
    "国家队公布最新一期足球集训名单",
    "A股市场持续震荡，投资者需谨慎",
    "篮球巨星刷新历史得分记录",
    "理财产品收益率创下新高"
]
headlines_tokenized = [jieba.lcut(doc) for doc in headlines]
dictionary = corpora.Dictionary(headlines_tokenized)
headlines_bow = [dictionary.doc2bow(doc) for doc in headlines_tokenized]
lda_model = models.LdaModel(corpus=headlines_bow, num_topics=2, id2word=dictionary, random_state=100)
for i in range(2):
    print(f"主题 {i}:")
    print(lda_model.print_topic(i))
    
new_headline = "巨星詹姆斯获得常规赛MVP"
new_tokenized = jieba.lcut(new_headline)
new_bow = dictionary.doc2bow(new_tokenized)
topic_distribution = lda_model[new_bow]
print("新标题的主题分布:")
print(topic_distribution)

主题 0:
0.045*"，" + 0.040*"公布" + 0.039*"一期" + 0.039*"名单" + 0.039*"足球" + 0.039*"最新" + 0.038*"集训" + 0.038*"国家队" + 0.037*"A股" + 0.037*"市场"
主题 1:
0.066*"，" + 0.039*"篮球" + 0.039*"刷新" + 0.039*"历史" + 0.039*"记录" + 0.038*"得分" + 0.038*"巨星" + 0.037*"刺激" + 0.036*"降息" + 0.036*"反弹"
新标题的主题分布:
[(0, np.float32(0.27243596)), (1, np.float32(0.72756404))]


# Word2Vec
## 直接输入分词后的列表

In [13]:
from gensim.models import Word2Vec
headlines = [
    # 财经
    "央行降息，刺激股市反弹",
    "A股市场持续震荡，投资者需谨慎",
    "理财产品收益率创下新高",
    "证监会发布新规，规范市场交易",
    "创业板指数上涨，科技股领涨大盘",
    "房价调控政策出台，房地产市场降温",
    "全球股市动荡，影响资本市场信心",
    "分析师认为，当前股市风险与机遇并存，市场情绪复杂",

    # 体育
    "球队赢得总决赛冠军，球员表现出色",
    "国家队公布最新一期足球集训名单",
    "篮球巨星刷新历史得分记录",
    "奥运会开幕，中国代表团旗手确定",
    "马拉松比赛圆满结束，选手创造佳绩",
    "电子竞技联赛吸引大量年轻观众",
    "这支球队的每位球员都表现出色",
    "球员转会市场活跃，多支球队积极引援"
]
headlines_tokenized = [jieba.lcut(doc) for doc in headlines]
model = Word2Vec(headlines_tokenized,vector_size=50,window=3,min_count=1,sg=1)
#训练完成后，词向量存储在model.wv
#window为窗口大小
#min_count为词频阈值，低于该阈值的词语会被忽略
#sg为训练算法，1表示skip-gram，0表示CBOW
#hs表示选择优化策略。0 表示使用 Negative Sampling (负采样)；1 表示使用 Hierarchical Softmax。当 hs=0 时，下面的 negative 参数才会生效。
#negative表示负采样的样本数，通常设置为5-20之间，默认值为5
#sample表示高频词汇的随机下采样阈值

similar_to_market = model.wv.most_similar("股市")
print("与'股市'最相似的词语：")
for word, similarity in similar_to_market:
    print(f"{word}: {similarity:.4f}")
    
similarity = model.wv.similarity("球队", "球员")
print(f"'球队'与'球员'的相似度：{similarity:.4f}")

market_vector = model.wv["市场"]
print(f"'市场'的词向量：{market_vector}")

from gensim.models import KeyedVectors
model.wv.save("news_vectors.kv")
loaded_wv = KeyedVectors.load("news_vectors.kv")
print("loaded_wv.similarity('股市', '市场'):", loaded_wv.similarity("股市", "市场"))


与'股市'最相似的词语：
赢得: 0.3200
的: 0.3075
出台: 0.2861
领涨: 0.2767
大量: 0.2701
房价: 0.2357
指数: 0.2333
交易: 0.2314
活跃: 0.2247
反弹: 0.2169
'球队'与'球员'的相似度：-0.1547
'市场'的词向量：[-0.01629895  0.00898576 -0.00825693  0.00171046  0.01693705 -0.00894291
  0.00908277 -0.01354792 -0.00709458  0.01877438 -0.00308243  0.00057008
 -0.00823103 -0.01533619 -0.00306791  0.00488128 -0.00174436  0.0110105
 -0.0054841   0.00452341  0.01097944  0.01677649 -0.00285141 -0.01840115
  0.00880808  0.00116408  0.0148045  -0.00154266 -0.00534122 -0.01753996
 -0.00171906  0.00558483  0.01087289  0.01402907 -0.01141106  0.00370407
  0.0121832  -0.0095658  -0.00625429  0.01362406  0.0032042   0.00036285
  0.00689516  0.00044014  0.01924145  0.01006901 -0.01783963 -0.01406283
  0.00177821  0.01283124]
loaded_wv.similarity('股市', '市场'): 0.017658304
